In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("/content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource/site_trial_history_extended.csv")

cat_cols = ["protocol_id", "country", "region", "therapeutic_area", "phase",
            "sponsor_type", "regulatory_authority_type", "ec_type"]

num_cols = [
    "protocol_complexity_score", "num_visits",
    "num_inclusion_criteria", "num_exclusion_criteria",
    "biomarker_required", "prior_therapy_restriction", "placebo_controlled",
    "pi_experience_years", "pi_prior_trials_same_indication",
    "coordinator_count", "gcp_training_valid",
    "competing_trials_same_indication_region",
    "eligible_patient_pool_estimate",
    "expected_duration_years",
    "fpi_delay_days", "recruitment_window_days", "schedule_slip_days",
    "start_quarter",
    "regulatory_delay_days", "import_license_required",
    "ec_approval_delay_days", "ec_submission_to_fpi_days", "num_ec_amendments",
    "adv_poster", "adv_pamphlet", "adv_email", "adv_online",
    "adv_newsletter", "adv_newspaper", "adv_radio", "adv_tv",
    "adv_community_presentation", "adv_other",
    "num_active_strategies", "num_passive_strategies", "total_strategies",
    "active_outreach_flag", "passive_only_flag", "digital_flag", "ad_cost_total",
    "screen_failure_rate",
]

X = df[cat_cols + num_cols]
y = df["actual_enrolled"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])
model = Pipeline([
    ("prep", pre),
    ("gbr", GradientBoostingRegressor(n_estimators=500, learning_rate=0.05,
                                      max_depth=3, random_state=42)),
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(Xtr, ytr)
pred = model.predict(Xte)
print(f"MAE: {mean_absolute_error(yte, pred):.2f} subjects")
print(f"R² : {r2_score(yte, pred):.3f}")

joblib.dump(model, "site_enrollment_model.pkl")

MAE: 12.52 subjects
R² : 0.983


['site_enrollment_model.pkl']

In [3]:
import pandas as pd, numpy as np, joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

df = pd.read_csv("/content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource/site_trial_history_extended.csv")

# Label = best channel (what we want to predict)
df = df[df["best_channel"] != "none"].copy()

cat_cols = ["protocol_id", "country", "region", "therapeutic_area",
            "phase", "sponsor_type", "ec_type", "regulatory_authority_type"]
num_cols = [
    "protocol_complexity_score", "biomarker_required",
    "eligible_patient_pool_estimate",
    "competing_trials_same_indication_region",
    "pi_experience_years", "pi_prior_trials_same_indication",
    "coordinator_count",
    "fpi_delay_days", "recruitment_window_days",
    "ec_approval_delay_days", "num_ec_amendments",
    "screen_failure_rate",
]

X = df[cat_cols + num_cols]
y = df["best_channel"]

pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])
clf = Pipeline([
    ("prep", pre),
    ("rf", RandomForestClassifier(n_estimators=500, max_depth=12,
                                  random_state=42, n_jobs=-1)),
])

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                      random_state=42, stratify=y)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)

print(f"Accuracy: {accuracy_score(yte, pred):.3f}")
print("\nClassification report:")
print(classification_report(yte, pred))

joblib.dump(clf, "best_channel_model.pkl")

# ----------------------------------------------------------------------
# Recommend best strategy for each protocol for a NEW hypothetical site
# ----------------------------------------------------------------------
new_site = {
    "country": "Germany", "region": "Europe",
    "therapeutic_area": None, "phase": None, "sponsor_type": None,
    "ec_type": "CEC", "regulatory_authority_type": "EMA",
    "protocol_complexity_score": None, "biomarker_required": None,
    "eligible_patient_pool_estimate": 350,
    "competing_trials_same_indication_region": 3,
    "pi_experience_years": 12, "pi_prior_trials_same_indication": 4,
    "coordinator_count": 3,
    "fpi_delay_days": 90, "recruitment_window_days": 540,
    "ec_approval_delay_days": 60, "num_ec_amendments": 1,
    "screen_failure_rate": 0.35,
}

PROTOCOLS = {
    "PROTO-A_ONCO": dict(therapeutic_area="Oncology", phase="III",
                         sponsor_type="Pharma", protocol_complexity_score=8,
                         biomarker_required=1),
    "PROTO-B_CARDIO": dict(therapeutic_area="Cardiology", phase="II",
                           sponsor_type="Biotech", protocol_complexity_score=5,
                           biomarker_required=0),
    "PROTO-C_NEURO": dict(therapeutic_area="Neurology", phase="II",
                          sponsor_type="Academic", protocol_complexity_score=7,
                          biomarker_required=1),
}

rows = []
for pid, p in PROTOCOLS.items():
    row = {**new_site, **p, "protocol_id": pid}
    rows.append(row)
new_df = pd.DataFrame(rows)

proba = clf.predict_proba(new_df[cat_cols + num_cols])
classes = clf.classes_
best_idx = proba.argmax(axis=1)

new_df["recommended_channel"] = classes[best_idx]
new_df["confidence"] = proba.max(axis=1).round(2)

print("\n🎯 Recommended advertisement strategy per protocol:")
print(new_df[["protocol_id", "recommended_channel", "confidence"]].to_string(index=False))

Accuracy: 0.455

Classification report:
                            precision    recall  f1-score   support

adv_community_presentation       0.31      0.31      0.31       121
                 adv_email       0.49      0.85      0.62       274
            adv_newsletter       0.00      0.00      0.00        22
             adv_newspaper       0.00      0.00      0.00         1
                adv_online       0.00      0.00      0.00        95
                 adv_other       0.00      0.00      0.00        14
              adv_pamphlet       0.00      0.00      0.00        10
                adv_poster       0.00      0.00      0.00        58
                 adv_radio       0.00      0.00      0.00         1

                  accuracy                           0.45       596
                 macro avg       0.09      0.13      0.10       596
              weighted avg       0.29      0.45      0.35       596



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



🎯 Recommended advertisement strategy per protocol:
   protocol_id        recommended_channel  confidence
  PROTO-A_ONCO                  adv_email        0.34
PROTO-B_CARDIO                  adv_email        0.56
 PROTO-C_NEURO adv_community_presentation        0.30


In [5]:
"""
slide_visuals.py
Generates all slide-ready plots and tables from the extended dataset.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import MaxNLocator

# Use a clean, presentation-friendly style
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "axes.titleweight": "bold",
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

# Brand colors (change to client palette)
BRAND = ["#1F4E79", "#2E75B6", "#9DC3E6", "#F4B183", "#C00000", "#548235", "#7030A0"]

# ---------------------------------------------------------------
# Load data
# ---------------------------------------------------------------
df = pd.read_csv("/content/drive/MyDrive/HealthCare_/CTR_Recruitment/DataSource/site_trial_history_extended.csv")
df["actual_first_patient_in"] = pd.to_datetime(df["actual_first_patient_in"])

# Convenience mappings
PROTO_LABEL = {
    "PROTO-A_ONCO":  "A · Oncology (NSCLC, Ph III)",
    "PROTO-B_CARDIO":"B · Cardiology (HF, Ph II)",
    "PROTO-C_NEURO": "C · Neurology (AD, Ph II)",
}
df["protocol_label"] = df["protocol_id"].map(PROTO_LABEL)

CHANNELS = ["adv_poster","adv_pamphlet","adv_email","adv_online",
            "adv_newsletter","adv_newspaper","adv_radio","adv_tv",
            "adv_community_presentation","adv_other"]
CHANNEL_LABEL = {
    "adv_poster":"Poster","adv_pamphlet":"Pamphlet","adv_email":"Email",
    "adv_online":"Online","adv_newsletter":"Newsletter","adv_newspaper":"Newspaper",
    "adv_radio":"Radio","adv_tv":"TV","adv_community_presentation":"Community",
    "adv_other":"Other",
}
df["channel_short"] = df["best_channel"].map(CHANNEL_LABEL).fillna("None")

In [6]:
overview = pd.DataFrame({
    "Dimension": [
        "Total trial records", "Protocols covered", "Countries",
        "Regulatory authorities", "Ethics committee types",
        "Advertisement channels tracked",
        "Timeline features", "Regulatory features", "Derived features",
        "Target variables",
    ],
    "Value": [
        f"{len(df):,}", df['protocol_id'].nunique(), df['country'].nunique(),
        df['regulatory_authority_type'].nunique(), df['ec_type'].nunique(),
        len(CHANNELS), 10, 11, 21, "2 (enrollment, best channel)",
    ],
})
overview.to_csv("slide1_overview.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis("off")
tbl = ax.table(cellText=overview.values, colLabels=overview.columns,
               cellLoc="left", colLoc="left", loc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(12)
tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(BRAND[0]); cell.set_text_props(color="white", weight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#F2F7FB")
plt.title("Model Dataset at a Glance", pad=20, weight="bold")
plt.savefig("slide1_overview.png")
plt.close()

In [7]:
fig, ax = plt.subplots(figsize=(10, 6))
order = list(PROTO_LABEL.values())
sns.boxplot(data=df, x="protocol_label", y="actual_enrolled",
            order=order, palette=BRAND[:3], ax=ax, width=0.55)
sns.stripplot(data=df, x="protocol_label", y="actual_enrolled",
              order=order, color="black", alpha=0.15, size=2, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Subjects enrolled")
ax.set_title("Enrollment distribution by protocol")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
plt.xticks(rotation=10)
plt.savefig("slide2_enrollment_box.png")
plt.close()

# Summary table
summary = df.groupby("protocol_label")["actual_enrolled"].agg(
    ["count","mean","median","std","min","max"]
).round(1).reset_index()
summary.to_csv("slide2_enrollment_summary.csv", index=False)

/tmp/ipykernel_2912/115315102.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="protocol_label", y="actual_enrolled",


In [8]:
fig, ax = plt.subplots(figsize=(10, 6))
for i, (pid, grp) in enumerate(df.groupby("protocol_label")):
    ax.scatter(grp["recruitment_window_days"], grp["actual_enrolled"],
               alpha=0.45, s=22, color=BRAND[i], label=pid)
    # trend line
    z = np.polyfit(grp["recruitment_window_days"], grp["actual_enrolled"], 1)
    xs = np.linspace(grp["recruitment_window_days"].min(),
                     grp["recruitment_window_days"].max(), 100)
    ax.plot(xs, np.polyval(z, xs), color=BRAND[i], lw=2)
ax.set_xlabel("Recruitment window (days, FPI → LPI)")
ax.set_ylabel("Subjects enrolled")
ax.set_title("Longer recruitment windows yield more subjects")
ax.legend(title="Protocol", frameon=True)
plt.savefig("slide3_window_vs_enrollment.png")
plt.close()

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: FPI delay distribution
sns.boxplot(data=df, x="ec_type", y="fpi_delay_days",
            palette=BRAND[:3], ax=axes[0], width=0.5)
axes[0].set_title("FPI delay by ethics committee type")
axes[0].set_xlabel("")
axes[0].set_ylabel("Days from estimated start to actual FPI")

# Right: Approval delay distribution
sns.boxplot(data=df, x="ec_type", y="ec_approval_delay_days",
            palette=BRAND[:3], ax=axes[1], width=0.5)
axes[1].set_title("Ethics approval delay by committee type")
axes[1].set_xlabel("")
axes[1].set_ylabel("Days from submission to approval")

plt.tight_layout()
plt.savefig("slide4_ec_impact.png")
plt.close()

# Table
ec_tbl = df.groupby("ec_type").agg(
    n=("trial_id","count"),
    mean_fpi_delay=("fpi_delay_days","mean"),
    median_fpi_delay=("fpi_delay_days","median"),
    mean_approval_delay=("ec_approval_delay_days","mean"),
    mean_amendments=("num_ec_amendments","mean"),
).round(1).reset_index()
ec_tbl.to_csv("slide4_ec_summary.csv", index=False)

/tmp/ipykernel_2912/4199609056.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="ec_type", y="fpi_delay_days",
/tmp/ipykernel_2912/4199609056.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="ec_type", y="ec_approval_delay_days",


In [10]:
usage = df.groupby("protocol_label")[CHANNELS].mean().round(2)
usage.columns = [CHANNEL_LABEL[c] for c in usage.columns]

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(usage, annot=True, fmt=".0%", cmap="Blues",
            cbar_kws={"label": "% of trials using channel"}, ax=ax)
ax.set_title("Channel usage rate by protocol")
ax.set_xlabel("")
ax.set_ylabel("")
plt.savefig("slide5_channel_usage.png")
plt.close()
usage.to_csv("slide5_channel_usage.csv")

In [11]:
# Effectiveness = mean enrollment when channel used vs. not used
effect = []
for pid, grp in df.groupby("protocol_label"):
    for ch in CHANNELS:
        used = grp[grp[ch] == 1]["actual_enrolled"].mean()
        not_used = grp[grp[ch] == 0]["actual_enrolled"].mean()
        effect.append({
            "Protocol": pid,
            "Channel": CHANNEL_LABEL[ch],
            "Lift": (used - not_used) / not_used * 100 if not_used > 0 else 0,
        })
effect_df = pd.DataFrame(effect)
pivot = effect_df.pivot(index="Channel", columns="Protocol", values="Lift")

fig, ax = plt.subplots(figsize=(11, 6))
sns.heatmap(pivot, annot=True, fmt=".0f", cmap="RdYlGn", center=0,
            cbar_kws={"label": "% lift in enrollment"}, ax=ax)
ax.set_title("Enrollment lift by channel and protocol")
ax.set_xlabel("")
ax.set_ylabel("")
plt.savefig("slide6_channel_effectiveness.png")
plt.close()
pivot.round(1).to_csv("slide6_channel_effectiveness.csv")

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Boxplot active vs passive
df["outreach_type"] = np.where(df["active_outreach_flag"] == 1,
                               "Active outreach", "Passive only")
sns.boxplot(data=df, x="outreach_type", y="actual_enrolled",
            palette=[BRAND[4], BRAND[0]], ax=axes[0], width=0.5)
axes[0].set_title("Enrollment by outreach type")
axes[0].set_xlabel("")
axes[0].set_ylabel("Subjects enrolled")

# Right: Bar of mean by number of active strategies
strat = df.groupby("num_active_strategies")["actual_enrolled"].mean().reset_index()
sns.barplot(data=strat, x="num_active_strategies", y="actual_enrolled",
            palette="Blues_d", ax=axes[1])
axes[1].set_title("Diminishing returns with more active channels")
axes[1].set_xlabel("Number of active channels used")
axes[1].set_ylabel("Mean subjects enrolled")
for i, v in enumerate(strat["actual_enrolled"]):
    axes[1].text(i, v + 2, f"{v:.0f}", ha="center", fontsize=11, weight="bold")

plt.tight_layout()
plt.savefig("slide7_active_vs_passive.png")
plt.close()

active_tbl = df.groupby("outreach_type")["actual_enrolled"].agg(
    ["count","mean","median","std"]
).round(1).reset_index()
active_tbl.to_csv("slide7_active_summary.csv", index=False)

/tmp/ipykernel_2912/153794723.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x="outreach_type", y="actual_enrolled",
/tmp/ipykernel_2912/153794723.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=strat, x="num_active_strategies", y="actual_enrolled",


In [13]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

cat_cols = ["protocol_id","country","region","therapeutic_area","phase",
            "sponsor_type","regulatory_authority_type","ec_type"]
num_cols = [
    "protocol_complexity_score","num_visits","num_inclusion_criteria",
    "num_exclusion_criteria","biomarker_required","prior_therapy_restriction",
    "placebo_controlled","pi_experience_years","pi_prior_trials_same_indication",
    "coordinator_count","gcp_training_valid","competing_trials_same_indication_region",
    "eligible_patient_pool_estimate","expected_duration_years","fpi_delay_days",
    "recruitment_window_days","schedule_slip_days","start_quarter",
    "regulatory_delay_days","import_license_required","ec_approval_delay_days",
    "ec_submission_to_fpi_days","num_ec_amendments","adv_poster","adv_pamphlet",
    "adv_email","adv_online","adv_newsletter","adv_newspaper","adv_radio",
    "adv_tv","adv_community_presentation","adv_other","num_active_strategies",
    "num_passive_strategies","total_strategies","active_outreach_flag",
    "passive_only_flag","digital_flag","ad_cost_total","screen_failure_rate",
]
X = df[cat_cols + num_cols]; y = df["actual_enrolled"]

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                         ("num", "passthrough", num_cols)])
model = Pipeline([("prep", pre),
                  ("gbr", GradientBoostingRegressor(n_estimators=500,
                                                    learning_rate=0.05,
                                                    max_depth=3, random_state=42))])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(Xtr, ytr)

# Get importances back in original feature space
ohe = model.named_steps["prep"].named_transformers_["cat"]
cat_names = ohe.get_feature_names_out(cat_cols)
all_names = list(cat_names) + num_cols
importances = model.named_steps["gbr"].feature_importances_

imp_df = (pd.DataFrame({"Feature": all_names, "Importance": importances})
            .sort_values("Importance", ascending=False).head(15))

fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(data=imp_df, y="Feature", x="Importance", palette="Blues_r", ax=ax)
ax.set_title("Top 15 drivers of site enrollment")
ax.set_xlabel("Relative importance")
ax.set_ylabel("")
plt.savefig("slide8_feature_importance.png")
plt.close()
imp_df.to_csv("slide8_feature_importance.csv", index=False)

/tmp/ipykernel_2912/4183622003.py:45: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=imp_df, y="Feature", x="Importance", palette="Blues_r", ax=ax)


In [14]:
from sklearn.metrics import mean_absolute_error, r2_score
pred = model.predict(Xte)

fig, ax = plt.subplots(figsize=(8.5, 8))
ax.scatter(yte, pred, alpha=0.4, s=25, color=BRAND[0], edgecolor="white")
lim = [0, max(yte.max(), pred.max()) * 1.05]
ax.plot(lim, lim, "--", color=BRAND[4], lw=2, label="Perfect prediction")
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("Actual subjects enrolled")
ax.set_ylabel("Predicted subjects enrolled")
ax.set_title(f"Predicted vs. actual (R² = {r2_score(yte,pred):.2f}, "
             f"MAE = {mean_absolute_error(yte,pred):.0f})")
ax.legend()
plt.savefig("slide9_pred_vs_actual.png")
plt.close()

pd.DataFrame({
    "Metric": ["R²","MAE (subjects)","Training rows","Test rows"],
    "Value": [round(r2_score(yte,pred),3),
              round(mean_absolute_error(yte,pred),1),
              len(Xtr), len(Xte)],
}).to_csv("slide9_model_accuracy.csv", index=False)

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------------------
# 1. Load catalog and master dataset
# ----------------------------------------------------------------------
CATALOG_PATH = (
    "/content/drive/MyDrive/HealthCare_/CTR_Recruitment/"
    "DataSource/advertisement_strategy_catalog.csv"
)
DATA_PATH = (
    "/content/drive/MyDrive/HealthCare_/CTR_Recruitment/"
    "DataSource/site_trial_history_extended.csv"
)

catalog = pd.read_csv(CATALOG_PATH, index_col="channel")
df = pd.read_csv(DATA_PATH)

# Sanity checks
expected_channels = set(catalog.index)
missing = expected_channels - set(CHANNELS)
assert not missing, f"Channels missing from master df: {missing}"
assert "cost" in catalog.columns, "Catalog must have a 'cost' column"

# ----------------------------------------------------------------------
# 2. Compute lift-based cost efficiency
# ----------------------------------------------------------------------
rows = []
for ch in CHANNELS:
    used_mask = df[ch] == 1
    n_used = int(used_mask.sum())
    if n_used == 0:
        continue  # skip channels never used

    mean_used = df.loc[used_mask, "actual_enrolled"].mean()
    mean_not_used = df.loc[~used_mask, "actual_enrolled"].mean()
    lift = mean_used - mean_not_used
    cost = float(catalog.loc[ch, "cost"])

    rows.append({
        "Channel": CHANNEL_LABEL[ch],
        "Channel_key": ch,
        "N trials": n_used,
        "Mean enrolled (used)": round(mean_used, 1),
        "Mean enrolled (not used)": round(mean_not_used, 1),
        "Lift (subjects)": round(lift, 1),
        "Cost (USD)": cost,
        "Lift per $1,000": round(lift / cost * 1000, 2) if cost > 0 else np.nan,
    })

cost_df = pd.DataFrame(rows).sort_values("Lift per $1,000", ascending=False)

# ----------------------------------------------------------------------
# 3. Plot
# ----------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 6))
sns.barplot(data=cost_df, x="Lift per $1,000", y="Channel",
            palette="Greens_r", ax=ax)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Cost-efficiency of recruitment channels\n"
             "(incremental subjects enrolled per $1,000 spent)")
ax.set_xlabel("Additional subjects per $1,000")
ax.set_ylabel("")

# Annotate values
for i, (_, row) in enumerate(cost_df.iterrows()):
    ax.text(row["Lift per $1,000"], i,
            f"  {row['Lift per $1,000']:.2f}",
            va="center", fontsize=10)

plt.tight_layout()
plt.savefig("slide11_cost_efficiency.png", dpi=300, bbox_inches="tight")
plt.close()

# ----------------------------------------------------------------------
# 4. Save table
# ----------------------------------------------------------------------
cost_df.to_csv("slide11_cost_efficiency.csv", index=False)
print("✅ slide11_cost_efficiency.png / .csv saved")
print(cost_df.to_string(index=False))

/tmp/ipykernel_2912/3112266641.py:59: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=cost_df, x="Lift per $1,000", y="Channel",


✅ slide11_cost_efficiency.png / .csv saved
   Channel                Channel_key  N trials  Mean enrolled (used)  Mean enrolled (not used)  Lift (subjects)  Cost (USD)  Lift per $1,000
     Email                  adv_email      1532                 203.7                     159.2             44.5       500.0            89.00
 Community adv_community_presentation      1060                 204.3                     169.7             34.6       400.0            86.55
    Online                 adv_online      1282                 205.8                     164.2             41.6      1200.0            34.67
Newsletter             adv_newsletter       783                 187.5                     180.0              7.5       300.0            25.14
  Pamphlet               adv_pamphlet      1958                 182.4                     181.2              1.2       150.0             8.17
 Newspaper              adv_newspaper       606                 185.7                     181.0          

In [19]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

df_cls = df[df["best_channel"] != "none"].copy()
Xc = df_cls[cat_cols + num_cols]; yc = df_cls["best_channel"]

pre_c = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
                           ("num", "passthrough", num_cols)])
clf = Pipeline([("prep", pre_c),
                ("rf", RandomForestClassifier(n_estimators=500, max_depth=12,
                                              random_state=42, n_jobs=-1))])
Xtr_c, Xte_c, ytr_c, yte_c = train_test_split(Xc, yc, test_size=0.2,
                                              random_state=42, stratify=yc)
clf.fit(Xtr_c, ytr_c)
acc = accuracy_score(yte_c, clf.predict(Xte_c))

# Build recommendations for a representative new site
new_site = dict(country="Germany", region="Europe", ec_type="CEC",
                regulatory_authority_type="EMA",
                eligible_patient_pool_estimate=350,
                competing_trials_same_indication_region=3,
                pi_experience_years=12, pi_prior_trials_same_indication=4,
                coordinator_count=3, fpi_delay_days=90,
                recruitment_window_days=540, ec_approval_delay_days=60,
                num_ec_amendments=1, screen_failure_rate=0.35)

PROTO = {
    "PROTO-A_ONCO":  dict(therapeutic_area="Oncology", phase="III",
                          sponsor_type="Pharma", protocol_complexity_score=8,
                          biomarker_required=1, protocol_label=PROTO_LABEL["PROTO-A_ONCO"]),
    "PROTO-B_CARDIO":dict(therapeutic_area="Cardiology", phase="II",
                          sponsor_type="Biotech", protocol_complexity_score=5,
                          biomarker_required=0, protocol_label=PROTO_LABEL["PROTO-B_CARDIO"]),
    "PROTO-C_NEURO": dict(therapeutic_area="Neurology", phase="II",
                          sponsor_type="Academic", protocol_complexity_score=7,
                          biomarker_required=1, protocol_label=PROTO_LABEL["PROTO-C_NEURO"]),
}
rows = []
for pid, p in PROTO.items():
    rows.append({**new_site, **p, "protocol_id": pid})
new_df = pd.DataFrame(rows)
proba = clf.predict_proba(new_df[cat_cols + num_cols])
classes = clf.classes_
top3 = np.argsort(-proba, axis=1)[:, :3]

rec_rows = []
for i, pid in enumerate(new_df["protocol_id"]):
    rec_rows.append({
        "Protocol": PROTO[pid]["protocol_label"],
        "Top channel": CHANNEL_LABEL[classes[top3[i,0]]],
        "Confidence": f"{proba[i, top3[i,0]]:.0%}",
        "2nd choice": CHANNEL_LABEL[classes[top3[i,1]]],
        "3rd choice": CHANNEL_LABEL[classes[top3[i,2]]],
    })
rec_df = pd.DataFrame(rec_rows)
rec_df.to_csv("slide10_recommendations.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis("off")
tbl = ax.table(cellText=rec_df.values, colLabels=rec_df.columns,
               cellLoc="left", colLoc="left", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(12); tbl.scale(1, 1.9)
for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor(BRAND[0]); cell.set_text_props(color="white", weight="bold")
    elif r % 2 == 0:
        cell.set_facecolor("#F2F7FB")
plt.title(f"Recommended recruitment strategy per protocol "
          f"(model accuracy = {acc:.0%})", pad=18, weight="bold")
plt.savefig("slide10_recommendations.png")
plt.close()

KeyError: "['num_visits', 'num_inclusion_criteria', 'num_exclusion_criteria', 'prior_therapy_restriction', 'placebo_controlled', 'gcp_training_valid', 'expected_duration_years', 'schedule_slip_days', 'start_quarter', 'regulatory_delay_days', 'import_license_required', 'ec_submission_to_fpi_days', 'adv_poster', 'adv_pamphlet', 'adv_email', 'adv_online', 'adv_newsletter', 'adv_newspaper', 'adv_radio', 'adv_tv', 'adv_community_presentation', 'adv_other', 'num_active_strategies', 'num_passive_strategies', 'total_strategies', 'active_outreach_flag', 'passive_only_flag', 'digital_flag', 'ad_cost_total'] not in index"